<a id="top"></a>
# IE 643 · Chart-to-Table Extraction — Fully-Supervised Baseline **v1**
**Frozen Donut encoder + 6-layer BART decoder trained from scratch** &nbsp;|&nbsp; Team NeuroFusion &nbsp;|&nbsp; Kaggle, GPU T4

**Task.** Reconstruct the full underlying data table of a chart image. This notebook is *Deliverable 3* of the project: the **fully-supervised baseline** (trained with the ground-truth tables), against which the weakly-supervised model (QA-only supervision) will later be compared.

**What "v1" means.** The vision encoder is the *document-pretrained* Swin encoder of Donut (frozen, never updated). The decoder is a BART decoder with **random initialisation**, trained with token-level cross-entropy on the serialised table. (v2, a separate notebook, replaces this decoder with Donut's own pretrained decoder.)

```
chart PNG ──► Donut Swin encoder (FROZEN) ──► 4,800 visual tokens ──► BART decoder (TRAINED) ──► "header | .. [ROW] row1 | .. [ROW] row2 | .."
```

## Table of contents
| # | Section | What it covers |
|---|---|---|
| 1 | [Environment](#s1) | pinned library versions, GPU check |
| 2 | [Configuration & paths](#s2) | run switches, dataset and checkpoint discovery |
| 3 | [Donut backbone & processor](#s3) | why this encoder, image preprocessing, tokenizer |
| 4 | [Dataset](#s4) | ChartQA layout, table serialisation, label construction, sanity checks |
| 5 | [Model architecture](#s5) | frozen encoder + decoder, parameter counts |
| 6 | [Loss function](#s6) | shift-right alignment, cross-entropy, random-init sanity check |
| 7 | [Training](#s7) | hyper-parameters, epoch history, resumable trainer |
| 8 | [Loading a trained checkpoint](#s8) | restore weights, teacher-forced sanity check |
| 9 | [Decoding](#s9) | KV-cached greedy generation, repetition guard |
| 10 | [Evaluation](#s10) | table parsing, metric definitions, validation run |
| 11 | [Results](#s11) | recorded numbers, loss and metric plots, observations |
| 12 | [Inference app](#s12) | Gradio demo for uploading a chart |
| 13 | [Reproducibility notes & caveats](#s13) | environment, seeds, known limitations |

## How to run this notebook
* **Default = safe.** Training is **off** (`RUN_TRAINING = False`) so *Save & Run All* never starts a multi-hour run. Evaluation runs on a 32-chart smoke subset.
* Attach as inputs: the ChartQA dataset and the checkpoint dataset(s) (`epoch_1.pt`, `epoch_2.pt`, `epoch_3.pt`). GPU T4 and Internet On (Donut weights are downloaded from Hugging Face).
* Switches are in [Section 2](#s2).

## Result at a glance (v1, epoch 3, 1,054 validation charts, greedy decoding)
| Val loss | Exact table match | Shape match | Cell F1 | Header acc. | First-column acc. | Numeric acc. (±1%) |
|---|---|---|---|---|---|---|
| 1.2578 | 0.0009 | 0.1641 | 0.1317 | 0.6094 | 0.3425 | 0.1396 |

<a id="s1"></a>
## 1. Environment  <small>[↑ top](#top)</small>

**Why pin versions?** The checkpoints were trained and evaluated with `transformers==4.48.3`. Newer releases change the tokenizer backend and the KV-cache classes, so the pins keep results reproducible.

| Package | Version used |
|---|---|
| Python | 3.13 (Kaggle default at the time) |
| PyTorch | 2.11.0 + cu128 |
| transformers | 4.48.3 |
| tokenizers | 0.21.1 |
| huggingface-hub | 0.36.2 |
| pandas | 2.2.3 |
| Pillow | 11.3.0 |

pip prints *dependency-conflict* warnings for unrelated preinstalled Kaggle packages (e.g. `nilearn`, `dopamine-rl`); they do not affect this notebook. If an import fails after installing, restart the kernel once and **skip** the install cell.

In [ ]:
%pip install -q "transformers==4.48.3" "tokenizers==0.21.1" "huggingface-hub==0.36.2" "pandas==2.2.3" "Pillow==11.3.0" sentencepiece accelerate

In [ ]:
import sys, torch, transformers, huggingface_hub, pandas
print("Python:", sys.version.split()[0])
print("PyTorch:", torch.__version__, "| CUDA available:", torch.cuda.is_available())
print("Transformers:", transformers.__version__, "| HF Hub:", huggingface_hub.__version__, "| Pandas:", pandas.__version__)
if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))

<a id="s2"></a>
## 2. Configuration & paths  <small>[↑ top](#top)</small>

**Run switches** control which expensive steps execute:

| Switch | Default | Meaning |
|---|---|---|
| `RUN_TRAINING` | `False` | `True` re-runs one full-data epoch (about 3.6 h on a T4) |
| `TRAIN_EPOCH` | `2` | which epoch to train when `RUN_TRAINING` is on (2 or 3) |
| `EVAL_EPOCH` | `3` | which saved checkpoint to evaluate |
| `EVAL_N` | `32` | validation charts to evaluate; `None` = all 1,054 (about 21 min) |
| `RUN_APP` | `False` | `True` launches the Gradio demo (blocks the cell) |

**Path discovery.** Instead of hard-coding Kaggle's mounted paths, the dataset root is found by searching for `train_human.json`, and checkpoints by file name. `/kaggle/input` is read-only, so everything we write goes to `/kaggle/working`.

In [ ]:
import os, glob

# ---------------- run switches ----------------
RUN_TRAINING = False   # True = (re)train one full epoch
TRAIN_EPOCH  = 2       # 2 or 3
EVAL_EPOCH   = 3       # checkpoint to evaluate / demo
EVAL_N       = 32      # None = whole validation set
RUN_APP      = False   # True = launch the Gradio app

WORK = "/kaggle/working"

# ---------------- dataset root ----------------
hits = glob.glob("/kaggle/input/**/train_human.json", recursive=True)
assert hits, "train_human.json not found under /kaggle/input; attach the ChartQA dataset"
dataset_root = os.path.dirname(os.path.dirname(hits[0]))     # parent of train/ val/ test/

# ---------------- checkpoints ----------------
def find_ckpt(name):
    found = glob.glob(f"{WORK}/{name}") + sorted(glob.glob(f"/kaggle/input/**/{name}", recursive=True))
    return found[0] if found else None

CKPT = {e: find_ckpt(f"epoch_{e}.pt") for e in (1, 2, 3)}
print("dataset_root:", dataset_root)
for e, p in CKPT.items():
    print(f"epoch_{e}.pt :", p or "not attached")

<a id="s3"></a>
## 3. Donut backbone & processor  <small>[↑ top](#top)</small>

**Why Donut's encoder.** The project requires a *frozen, document-/screenshot-pretrained* encoder. Donut (`naver-clova-ix/donut-base`) was pretrained to read text from document images without any OCR engine, so its Swin Transformer features keep the spatial detail and text-reading signal needed to read small axis labels and data labels from a chart. A natural-image encoder (ResNet, CLIP) lacks that.

**What the encoder produces.** A 4-stage Swin (depths 2/2/14/2) with total stride 32. For the processor's default 2560×1920 input this is an 80×60 grid = **4,800 visual tokens of dimension 1024**. The decoder will cross-attend to these tokens.

**The processor** (`DonutProcessor`) bundles two things:
* an *image processor* (resize/pad to 2560×1920, normalise → `pixel_values`),
* a *tokenizer* (SentencePiece/XLM-R style, vocabulary 57,525) with special tokens `<s>`=0 (BOS), `<pad>`=1, `</s>`=2 (EOS).

Only the **encoder** of the loaded `VisionEncoderDecoderModel` is used; Donut's own decoder is discarded in v1.

In [ ]:
from transformers import DonutProcessor, VisionEncoderDecoderModel

DONUT_ID = "naver-clova-ix/donut-base"
processor = DonutProcessor.from_pretrained(DONUT_ID)
donut = VisionEncoderDecoderModel.from_pretrained(DONUT_ID)
tok = processor.tokenizer

print("Encoder class        :", type(donut.encoder).__name__)
print("Encoder hidden size  :", donut.encoder.config.hidden_size)
print("Image size (processor):", processor.image_processor.size)
print("Tokenizer vocab size :", len(tok))
print("BOS / PAD / EOS ids  :", tok.bos_token_id, tok.pad_token_id, tok.eos_token_id)

<a id="s4"></a>
## 4. Dataset  <small>[↑ top](#top)</small>

### 4.1 ChartQA layout used here
```
ChartQA-Dataset/
  train|val|test/
    png/                      chart images,  <imgname>.png
    tables/                   ground-truth tables, <imgname>.csv   (same base name as the PNG)
    <split>_human.json        QA pairs written by humans     {imgname, query, label}
    <split>_augmented.json    machine-generated QA pairs     {imgname, query, label}
```
For this **fully-supervised baseline only the image and its CSV are used**. The QA JSON files are read just to *list which charts exist* (one chart has many questions, so names are de-duplicated). The QA `query`/`label` fields become important in the weakly-supervised model.

### 4.2 Table serialisation (the decoder's target text)
A CSV is flattened to one string: cells joined by `" | "`, rows joined by `" [ROW] "`, header first.
```
Entity | 2015 | 2016 [ROW] Nigeria | 38.75 | 41.35 [ROW] Africa | 33.86 | 33.9
```
The same format is used when parsing predictions back into tables ([Section 10](#s10)), so training and evaluation are consistent.

### 4.3 Label construction
* tokenise the string with the Donut tokenizer (adds `<s>` … `</s>`),
* truncate/pad to **512 tokens**,
* replace padding ids by **`-100`** so the loss ignores them.

`pixel_values` has shape `(3, 2560, 1920)`. Missing cells are written as empty strings (`fillna("")`).

In [ ]:
import json
from PIL import Image
from torch.utils.data import Dataset

class ChartQATableDataset(Dataset):
    """One item = (chart image tensor, tokenised serialised ground-truth table)."""
    def __init__(self, root, split, processor, max_target_length=512):
        self.root, self.split, self.processor = root, split, processor
        self.max_target_length = max_target_length
        split_dir = os.path.join(root, split)

        # 1) collect unique chart names from both QA files (one chart appears in many QA records)
        names = set()
        for suffix in ["human", "augmented"]:
            with open(os.path.join(split_dir, f"{split}_{suffix}.json")) as f:
                names.update(r["imgname"] for r in json.load(f))

        # 2) keep charts whose PNG and CSV both exist; sorted => deterministic order
        self.samples = []
        for n in sorted(names):
            ip = os.path.join(split_dir, "png", n)
            cp = os.path.join(split_dir, "tables", os.path.splitext(n)[0] + ".csv")
            if os.path.isfile(ip) and os.path.isfile(cp):
                self.samples.append((ip, cp))
        print(f"{split}: {len(self.samples)} chart-table pairs")

    def __len__(self): return len(self.samples)

    def __getitem__(self, idx):
        ip, cp = self.samples[idx]
        # image -> normalised tensor (3, 2560, 1920)
        image = Image.open(ip).convert("RGB")
        pv = self.processor(image, return_tensors="pt").pixel_values.squeeze(0)

        # CSV -> "header | ... [ROW] row | ... [ROW] ..."
        df = pd.read_csv(cp).fillna("")
        header = " | ".join(map(str, df.columns))
        rows = [" | ".join(map(str, r)) for r in df.itertuples(index=False, name=None)]
        text = header + (" [ROW] " + " [ROW] ".join(rows) if rows else "")

        # text -> token ids, fixed length 512, padding masked with -100
        tk = self.processor.tokenizer(text, max_length=self.max_target_length,
                                      padding="max_length", truncation=True, return_tensors="pt")
        labels = tk.input_ids.squeeze(0)
        labels[labels == self.processor.tokenizer.pad_token_id] = -100
        return {"pixel_values": pv, "labels": labels}

train_dataset = ChartQATableDataset(dataset_root, "train", processor)
val_dataset   = ChartQATableDataset(dataset_root, "val", processor)
# The v1 runs used 18,316 train / 1,054 val charts.

**Sanity checks on the data.** The cell below confirms (a) train and validation share **no** charts (no leakage), (b) the label tensor looks as designed (starts with BOS, ends with EOS, padded with −100), and (c) the tokenizer round-trips the serialised table exactly.

In [ ]:
# (a) no train/val leakage
tr = {os.path.basename(p) for p, _ in train_dataset.samples}
va = {os.path.basename(p) for p, _ in val_dataset.samples}
print("train/val image overlap:", len(tr & va), "(must be 0)")

# (b) label structure of one item
item = val_dataset[0]
lab = item["labels"]; valid = lab[lab != -100]
print("pixel_values:", tuple(item["pixel_values"].shape), "| labels:", tuple(lab.shape))
print("valid tokens:", valid.numel(), "| first id:", valid[0].item(), "(BOS) | last id:", valid[-1].item(), "(EOS)")
print("padding positions (-100):", (lab == -100).sum().item())

# (c) tokenizer round trip on the serialised target
decoded = tok.decode(valid, skip_special_tokens=True)
print("\nSerialised target (first 250 chars):\n", decoded[:250])

<a id="s5"></a>
## 5. Model architecture  <small>[↑ top](#top)</small>

| Component | Setting | Trainable? |
|---|---|---|
| **Encoder** | Donut Swin, hidden size 1024, output 4,800 tokens | **No**: `requires_grad=False`, kept in `eval()` |
| **Decoder** | `BartForCausalLM`, **6 layers**, 8 heads (head dim 128), FFN 4096, `d_model` 1024, learned positions up to 1024, vocab 57,525, dropout 0.1 (BART default), **random init** | **Yes** |
| **Cross-attention** | every decoder layer attends to the 4,800 encoder tokens (`is_decoder=True`, `add_cross_attention=True`) | part of the decoder |

**Design points**
* `d_model` is set to the encoder's hidden size (1024), so encoder outputs feed cross-attention directly, with no projection layer.
* The encoder call sits inside `torch.no_grad()`: no activations are stored for it, which saves a lot of memory at 2560×1920.
* Special-token ids come from the Donut tokenizer, so the decoder's vocabulary matches the label ids.
* `forward` shifts the labels right to build the decoder inputs (teacher forcing); the shift is explained in [Section 6](#s6).
* `use_cache=False` in training; the KV cache is used only in generation ([Section 9](#s9)).

In [ ]:
import torch, torch.nn as nn
from transformers import BartConfig, BartForCausalLM
from transformers.models.bart.modeling_bart import shift_tokens_right

class DonutBartFullySupervisedModel(nn.Module):
    def __init__(self, donut, processor):
        super().__init__()
        self.encoder = donut.encoder                    # document-pretrained Swin encoder
        self.tokenizer = processor.tokenizer
        for p in self.encoder.parameters():             # FREEZE the encoder
            p.requires_grad = False
        tokenizer = self.tokenizer
        cfg = BartConfig(
            vocab_size=len(tokenizer), d_model=self.encoder.config.hidden_size,
            decoder_layers=6, decoder_attention_heads=8, decoder_ffn_dim=4096,
            max_position_embeddings=1024, pad_token_id=tokenizer.pad_token_id,
            bos_token_id=tokenizer.bos_token_id, eos_token_id=tokenizer.eos_token_id,
            decoder_start_token_id=tokenizer.bos_token_id,
            is_decoder=True, add_cross_attention=True, use_cache=False,
        )
        self.decoder = BartForCausalLM(cfg)             # random initialisation

    def forward(self, pixel_values, labels):
        with torch.no_grad():                           # no gradients through the encoder
            enc = self.encoder(pixel_values).last_hidden_state          # (B, 4800, 1024)
        dec_in = shift_tokens_right(labels, self.decoder.config.pad_token_id,
                                    self.decoder.config.decoder_start_token_id)
        out = self.decoder(input_ids=dec_in, encoder_hidden_states=enc,
                           labels=labels, use_cache=False, return_dict=True)
        return out.loss, out.logits

model = DonutBartFullySupervisedModel(donut, processor).cuda()

n_total = sum(p.numel() for p in model.parameters())
n_train = sum(p.numel() for p in model.parameters() if p.requires_grad)
n_enc   = sum(p.numel() for p in model.encoder.parameters() if p.requires_grad)
print(f"total parameters    : {n_total/1e6:.1f} M")
print(f"trainable parameters: {n_train/1e6:.1f} M   (decoder only)")
print(f"encoder trainable   : {n_enc}   (must be 0)")

<a id="s6"></a>
## 6. Loss function  <small>[↑ top](#top)</small>

**Objective.** Token-level **cross-entropy** between the decoder's predicted distribution and the ground-truth table tokens, computed with teacher forcing:

$$\mathcal{L}=-\frac{1}{|\mathcal{T}|}\sum_{t\in\mathcal{T}}\log p_\theta\big(y_t\mid y_{<t},\,\text{image}\big),\qquad \mathcal{T}=\{t:\ y_t\neq -100\}$$

**Alignment (teacher forcing).** The decoder input at position *t* is the label at *t−1*; `shift_tokens_right` prepends the decoder-start token (BOS) and drops the last label:

```
labels      :  <s>  Entity  |   2015 ...  </s>  -100 -100
decoder_in  :  <s>   <s>   Entity  |  ...  ...   </s> (pad)
```
Because the labels themselves start with `<s>`, the decoder input starts `[<s>, <s>, ...]`. This repeated BOS is harmless and is reproduced at inference (generation starts from `<s>` and the model emits `<s>` first).

**Details**
* `ignore_index = -100` removes padding from the loss; the mean is over real tokens only.
* `BartForCausalLM` computes the loss *without* an internal shift, so the shift in `forward` is required. Shifting a second time would train the model to predict the wrong token.
* No label smoothing and no class weighting.
* **Reported train loss** = running mean of per-sample losses over the epoch (weights change during the epoch). **Reported val loss** = mean per-chart loss over the validation set.
* **Sanity check:** a randomly initialised decoder should give a loss near $\ln V=\ln 57525\approx 10.96$. The cell below shows this, so any trained checkpoint is easy to recognise (loss ≪ 10).

In [ ]:
import math
labels = val_dataset[0]["labels"][None].cuda()
dec_in = shift_tokens_right(labels, model.decoder.config.pad_token_id, model.decoder.config.decoder_start_token_id)
print("labels     :", labels[0, :8].tolist())
print("decoder_in :", dec_in[0, :8].tolist(), " <- shifted right, starts with BOS twice")

model.eval()
with torch.no_grad(), torch.autocast("cuda", dtype=torch.float16):
    loss, _ = model(pixel_values=val_dataset[0]["pixel_values"][None].cuda(), labels=labels)
print(f"\nloss of the UNTRAINED decoder: {loss.item():.3f}   (ln V = {math.log(len(tok)):.3f})")

<a id="s7"></a>
## 7. Training  <small>[↑ top](#top)</small>

### 7.1 Hyper-parameters
| Setting | Value |
|---|---|
| Optimizer | AdamW, lr 1e-4, weight decay 0.01 (PyTorch default), trainable parameters only |
| Batch size | 1 (a 2560×1920 image per step; fits a T4 with fp16) |
| LR schedule | constant |
| Precision | fp16 autocast + `GradScaler` |
| Gradient clipping | none in epochs 1–2; **max-norm 1.0 in epoch 3** |
| Dropout | 0.1 in the decoder; the encoder stays in `eval()` |
| Targets | max 512 tokens |

### 7.2 Epoch history
| Epoch | Where | Training data | Seed | Time | Train loss | Val loss |
|---|---|---|---|---|---|---|
| 1 | Colab T4 | random **half** of train (9,158 samples, subset seed 42, shuffled loader) | unseeded shuffle | 113 min | 2.1170 | 2.0344 * |
| 2 | Kaggle T4 | full train (18,316), resumed from `epoch_1.pt` | 2 | ≈ 3.6 h | 1.5546 | 1.5469 |
| 3 | Kaggle T4 | full train, resumed from `epoch_2.pt` | 3 | ≈ 3.6 h | 1.1731 | 1.2578 |

\* The epoch-1 validation loss was computed with the model left in `train()` mode (dropout active), so it is slightly pessimistic. Epochs 2 and 3 validate in `eval()` mode.

**Epoch 1 (Colab, reference only).** It used the loop below; it is *not* re-run here. The checkpoint `epoch_1.pt` is the starting point for epoch 2.
```python
def run_epoch(loader, training):
    model.train(); model.encoder.eval()              # encoder frozen -> eval mode
    for batch in loader:
        with torch.autocast("cuda", dtype=torch.float16):
            loss, _ = model(pixel_values=batch["pixel_values"].cuda(), labels=batch["labels"].cuda())
        if training:
            scaler.scale(loss).backward(); scaler.step(optimizer); scaler.update()
```

### 7.3 Resumable trainer for epochs 2 and 3 (this notebook)
Kaggle sessions are limited to about 9 h and a full epoch takes about 3.6 h, so the trainer is built to survive interruptions:
* a **fixed, seeded permutation** of the training set → the same sample order after a restart;
* every **1,500 steps** it saves `ckpt*_latest.pt` (decoder + optimizer + scaler + step + running loss), written to a temp file and renamed, so a crash cannot leave a half-written checkpoint;
* it stops itself at **8.5 h**, saves, and can be re-run in a new session: it resumes from the latest mid-epoch file, otherwise from the previous epoch's checkpoint;
* at the end of the epoch it validates on the whole validation set and writes `epoch_N.pt`.

> This single function is a refactor of the two cells actually run on Kaggle (epoch 2: seed 2, no clipping; epoch 3: seed 3, clip 1.0). The training logic is identical; only the epoch-specific settings were moved into `EPOCH_CFG`.

In [ ]:
import time, numpy as np
from torch.utils.data import DataLoader, Subset
from tqdm.auto import tqdm
os.environ["TOKENIZERS_PARALLELISM"] = "false"

# epoch-specific settings used in the original runs
EPOCH_CFG = {
    2: dict(seed=2, clip=None, init_from=1, latest="ckpt_latest.pt"),
    3: dict(seed=3, clip=1.0,  init_from=2, latest="ckpt3_latest.pt"),
}
SAVE_EVERY, TIME_BUDGET_H, LR = 1500, 8.5, 1e-4

def train_full_epoch(epoch):
    cfg = EPOCH_CFG[epoch]
    latest = f"{WORK}/{cfg['latest']}"
    trainable = [p for p in model.parameters() if p.requires_grad]
    optimizer = torch.optim.AdamW(trainable, lr=LR)
    scaler = torch.amp.GradScaler("cuda")

    # ---- choose where to resume from ----
    cands = ([latest] if os.path.exists(latest) else []) + glob.glob(f"/kaggle/input/**/{cfg['latest']}", recursive=True)
    src = cands[0] if cands else CKPT[cfg["init_from"]]
    assert src, f"no checkpoint to start from (need epoch_{cfg['init_from']}.pt or {cfg['latest']})"
    ck = torch.load(src, map_location="cpu", weights_only=False)
    if "step" in ck:                                    # mid-epoch checkpoint (decoder only)
        model.decoder.load_state_dict(ck["decoder_state_dict"])
        start_step, loss_sum = ck["step"], ck["loss_sum"]
    else:                                               # full checkpoint of the previous epoch
        model.load_state_dict(ck["model_state_dict"]); start_step, loss_sum = 0, 0.0
    optimizer.load_state_dict(ck["optimizer_state_dict"]); scaler.load_state_dict(ck["scaler_state_dict"])
    del ck
    print(f"Resuming from {src} at step {start_step}", flush=True)

    def save_latest(step, loss_sum):
        torch.save({"step": step, "loss_sum": loss_sum,
                    "decoder_state_dict": model.decoder.state_dict(),
                    "optimizer_state_dict": optimizer.state_dict(),
                    "scaler_state_dict": scaler.state_dict()}, latest + ".tmp")
        os.replace(latest + ".tmp", latest)             # atomic rename

    # ---- fixed sample order so a restart continues exactly where it stopped ----
    perm = np.random.default_rng(cfg["seed"]).permutation(len(train_dataset))
    loader = DataLoader(Subset(train_dataset, perm[start_step:].tolist()), batch_size=1,
                        shuffle=False, num_workers=2, pin_memory=True)

    model.train(); model.encoder.eval()
    step, t0, stopped = start_step, time.time(), False
    bar = tqdm(loader, total=len(perm) - start_step, desc=f"Epoch {epoch} (full data)")
    for batch in bar:
        pv = batch["pixel_values"].cuda(non_blocking=True)
        lb = batch["labels"].cuda(non_blocking=True)
        optimizer.zero_grad(set_to_none=True)
        with torch.autocast("cuda", dtype=torch.float16):
            loss, _ = model(pixel_values=pv, labels=lb)      # plain cross-entropy
        scaler.scale(loss).backward()
        if cfg["clip"]:                                      # epoch 3 only
            scaler.unscale_(optimizer)                       # clip the true (unscaled) gradients
            torch.nn.utils.clip_grad_norm_(trainable, cfg["clip"])
        scaler.step(optimizer); scaler.update()
        loss_sum += loss.item(); step += 1
        bar.set_postfix(train_loss=f"{loss_sum/step:.4f}")
        if step % SAVE_EVERY == 0: save_latest(step, loss_sum)
        if time.time() - t0 > TIME_BUDGET_H * 3600:
            save_latest(step, loss_sum); stopped = True; break

    if stopped:
        print(f"Time budget hit at step {step}/{len(perm)}. Re-run this cell in a new session.")
        return
    # ---- end of epoch: validate on all validation charts, then save the full checkpoint ----
    train_loss = loss_sum / step
    model.eval(); vs = 0.0
    with torch.no_grad():
        for b in tqdm(DataLoader(val_dataset, batch_size=1, num_workers=2), desc="Validation"):
            with torch.autocast("cuda", dtype=torch.float16):
                l, _ = model(pixel_values=b["pixel_values"].cuda(), labels=b["labels"].cuda())
            vs += l.item()
    val_loss = vs / len(val_dataset)
    print(f"Epoch {epoch} | train {train_loss:.4f} | val {val_loss:.4f}")
    torch.save({"epoch": epoch, "model_state_dict": model.state_dict(),
                "optimizer_state_dict": optimizer.state_dict(), "scaler_state_dict": scaler.state_dict(),
                "train_loss": train_loss, "val_loss": val_loss,
                "train_samples": len(train_dataset)}, f"{WORK}/epoch_{epoch}.pt")
    if os.path.exists(latest): os.remove(latest)

print("Trainer defined.")

**Launch cell.** Nothing runs unless `RUN_TRAINING = True` in [Section 2](#s2). When it does, it trains `TRAIN_EPOCH` on the full training set (about 3.6 h).

In [ ]:
if RUN_TRAINING:
    train_full_epoch(TRAIN_EPOCH)
else:
    print("RUN_TRAINING is False -> training skipped. Existing checkpoints are used below.")

<a id="s8"></a>
## 8. Loading a trained checkpoint  <small>[↑ top](#top)</small>

A checkpoint file is a dictionary with `model_state_dict` (encoder + decoder), `optimizer_state_dict`, `scaler_state_dict`, `epoch`, `train_loss`, `val_loss`. For evaluation only `model_state_dict` is needed; the optimizer state is what makes the file ≈ 2.2 GB.

**Check that the right weights were loaded.** The teacher-forced loss on one validation chart should be close to the recorded validation loss (a few units below 10.96); an untrained model would give about 11. For one chart the value is noisy; it only confirms that the weights are not random.

In [ ]:
path = CKPT[EVAL_EPOCH]
assert path, f"epoch_{EVAL_EPOCH}.pt not found; attach the checkpoint dataset"
ck = torch.load(path, map_location="cpu", weights_only=False)
model.load_state_dict(ck["model_state_dict"])
print(f"Loaded {path}\n  epoch {ck['epoch']} | recorded train loss {ck['train_loss']:.4f} | recorded val loss {ck['val_loss']:.4f}")
del ck
torch.cuda.empty_cache()
model.eval()

s = val_dataset[0]
with torch.no_grad(), torch.autocast("cuda", dtype=torch.float16):
    loss, _ = model(pixel_values=s["pixel_values"][None].cuda(), labels=s["labels"][None].cuda())
print("teacher-forced loss on val chart 0:", round(loss.item(), 4), "(about 11 would mean untrained weights)")

<a id="s9"></a>
## 9. Decoding  <small>[↑ top](#top)</small>

At inference there are no ground-truth tokens, so the table is produced token by token (**greedy decoding**: always take the highest-probability token).

**How the loop works**
1. Encode the image once → `hs` (4,800 visual tokens).
2. Start from the BOS token; at each step feed the newest token and take `argmax` of the last logits.
3. Stop at EOS (finished sequences are padded) or after `max_new_tokens = 512`.

**Why a manual loop.** It guarantees that every step cross-attends to the encoder states, and it uses the **KV cache** (`past_key_values`), so each step processes one new token instead of re-running the whole prefix. The library `generate()` on a bare decoder wrapper is not relied on here.

**Optional repetition guard** (`no_repeat_ngram = n`). Models like this one sometimes fall into loops (`| | | …`, `39 | 39 | …`). With the guard on, any token that would repeat an already-seen *n*-gram is banned. In the experiments the guard stopped loops but also blocked legitimate repeated patterns in tables, so it is reported as a **separate ablation**, and the plain greedy setting (`n = 0`) is the main result.

In [ ]:
import math, json

@torch.no_grad()
def greedy_generate_guarded(model, pixel_values, max_new_tokens=512, no_repeat_ngram=0):
    cfg = model.decoder.config
    hs = model.encoder(pixel_values).last_hidden_state              # encode image once
    B, dev, n = pixel_values.size(0), pixel_values.device, no_repeat_ngram
    cur = torch.full((B, 1), cfg.decoder_start_token_id, device=dev) # start with BOS
    out, past = [cur], None
    done = torch.zeros(B, dtype=torch.bool, device=dev)
    hist, seen = [[] for _ in range(B)], [dict() for _ in range(B)]  # for the n-gram guard
    for _ in range(max_new_tokens):
        o = model.decoder(input_ids=cur, encoder_hidden_states=hs,
                          past_key_values=past, use_cache=True, return_dict=True)
        past = o.past_key_values                                      # KV cache
        logits = o.logits[:, -1].float()
        if n > 1:                                                     # ban tokens that would repeat an n-gram
            for b in range(B):
                if done[b] or len(hist[b]) < n - 1: continue
                banned = seen[b].get(tuple(hist[b][-(n - 1):]))
                if banned: logits[b, list(banned)] = float("-inf")
        nxt = logits.argmax(-1)                                       # greedy choice
        nxt = torch.where(done, torch.full_like(nxt, cfg.pad_token_id), nxt)
        if n > 1:
            for b in range(B):
                if done[b]: continue
                if len(hist[b]) >= n - 1:
                    seen[b].setdefault(tuple(hist[b][-(n - 1):]), set()).add(nxt[b].item())
                hist[b].append(nxt[b].item())
        out.append(nxt[:, None]); done |= nxt == cfg.eos_token_id
        if done.all(): break
        cur = nxt[:, None]
    return torch.cat(out, 1)                                          # (B, 1 + generated length)

# quick demo on one validation chart
img_path, csv_path = val_dataset.samples[0]
pv = val_dataset[0]["pixel_values"][None].cuda()
with torch.autocast("cuda", dtype=torch.float16):
    gen = greedy_generate_guarded(model, pv, max_new_tokens=512)
ids = gen[0, 1:].tolist()
if tok.eos_token_id in ids: ids = ids[:ids.index(tok.eos_token_id)]
print("chart:", os.path.basename(img_path))
print("PRED:", tok.decode(ids, skip_special_tokens=True)[:300])

<a id="s10"></a>
## 10. Evaluation  <small>[↑ top](#top)</small>

### 10.1 Protocol
Greedy decoding (max 512 new tokens, fp16 autocast, batch 4) over the **validation charts**. Each generated string is parsed back into a table with the same `" | "` / `" [ROW] "` format and compared with the ground-truth CSV (serialised and parsed in the same way).

**Cell equality.** Two cells match if they are equal after whitespace normalisation and case-folding, **or** both are numbers that are equal (relative tolerance 1e-6; thousands separators are ignored).

**Matching is positional**, i.e. by `(row, column)`. A reference cell counts as correct only if the prediction has a cell at that exact position and the values match. This is strict: it does not reward the right value in the wrong place, and missing rows or columns count as errors. Tables of different size are handled naturally: extra predicted cells only reduce precision, missing cells reduce recall.

### 10.2 Metrics
| Metric | Definition |
|---|---|
| **Exact table match** | same shape *and* every reference cell correct |
| **Shape match** | same number of rows and same number of cells in every row |
| **Row-count match** | same number of rows |
| **Cell accuracy (micro)** | matched cells ÷ all reference cells (= recall) |
| **Cell accuracy (macro)** | mean over tables of (matched ÷ reference cells) |
| **Precision / F1** | matched ÷ predicted cells; F1 from micro precision and recall |
| **Header accuracy** | correct cells among the header row |
| **First-column accuracy** | correct cells in column 0, data rows only (entity names) |
| **Numeric accuracy (±1%)** | numeric data cells (row ≥ 1, column ≥ 1) predicted within 1% relative error |

Six validation references are longer than 512 tokens, so they can never be reproduced completely.

In [ ]:
import re
ROW_SPLIT  = re.compile(r"\s*\[ROW\]\s*")
CELL_SPLIT = re.compile(r"\s*\|\s*")

def table_to_text(csv_path):                       # same serialisation as the training target
    df = pd.read_csv(csv_path).fillna("")
    header = " | ".join(map(str, df.columns))
    rows = [" | ".join(map(str, r)) for r in df.itertuples(index=False, name=None)]
    return header + (" [ROW] " + " [ROW] ".join(rows) if rows else "")

def parse_table(text):                             # string -> list of rows, each a list of cells
    text = text.strip()
    return [CELL_SPLIT.split(r.strip()) for r in ROW_SPLIT.split(text)] if text else []

def norm(c): return re.sub(r"\s+", " ", str(c)).strip().casefold()
def to_num(c):
    try: return float(str(c).replace(",", ""))
    except (ValueError, TypeError): return None

def cells_equal(a, b):
    a, b = norm(a), norm(b)
    if a == b: return True
    x, y = to_num(a), to_num(b)
    return x is not None and y is not None and math.isclose(x, y, rel_tol=1e-6, abs_tol=1e-9)

def compare(pred, ref):                            # POSITIONAL matching
    n_ref, n_pred = sum(map(len, ref)), sum(map(len, pred))
    match = sum(1 for i, rrow in enumerate(ref) if i < len(pred)
                for j, rc in enumerate(rrow) if j < len(pred[i]) and cells_equal(pred[i][j], rc))
    shape = len(pred) == len(ref) and all(len(p) == len(r) for p, r in zip(pred, ref))
    return {"match": match, "n_ref": n_ref, "n_pred": n_pred, "shape": shape,
            "rows": len(pred) == len(ref), "exact": shape and match == n_ref}

def num_close(a, b, rel=0.01):                     # numeric tolerance of 1%
    x, y = to_num(a.strip()), to_num(b.strip())
    return x is not None and y is not None and math.isclose(x, y, rel_tol=rel, abs_tol=1e-9)

def extra_metrics(pred, ref):                      # header / first-column / numeric accuracy counters
    g = lambda i, j: pred[i][j] if i < len(pred) and j < len(pred[i]) else None
    hm = hn = em = en = nm = nn_ = 0
    for i, row in enumerate(ref):
        for j, rc in enumerate(row):
            pc = g(i, j)
            if i == 0:
                hn += 1; hm += pc is not None and cells_equal(pc, rc)
            elif j == 0:
                en += 1; em += pc is not None and cells_equal(pc, rc)
            elif to_num(rc.replace(",", "")) is not None:
                nn_ += 1; nm += pc is not None and num_close(pc, rc)
    return dict(hm=hm, hn=hn, em=em, en=en, nm=nm, nn=nn_)

# sanity check: a perfect prediction must score exact = True
ref = parse_table(table_to_text(val_dataset.samples[0][1]))
print(compare(ref, ref))

In [ ]:
@torch.no_grad()
def evaluate(tag, no_repeat_ngram=0, n_eval=None, bs=4):
    n = len(val_dataset) if n_eval is None else min(n_eval, len(val_dataset))
    loader = DataLoader(Subset(val_dataset, range(n)), batch_size=bs, shuffle=False,
                        num_workers=2, pin_memory=True)
    model.eval(); R, X, saved, idx, empty = [], [], [], 0, 0
    for batch in tqdm(loader, desc=f"Generating [{tag}]"):
        with torch.autocast("cuda", dtype=torch.float16):
            gen = greedy_generate_guarded(model, batch["pixel_values"].cuda(non_blocking=True),
                                          no_repeat_ngram=no_repeat_ngram)
        for row in gen:                                            # decode each chart in the batch
            ids = row[1:].tolist()
            if tok.eos_token_id in ids: ids = ids[:ids.index(tok.eos_token_id)]
            pt = tok.decode(ids, skip_special_tokens=True).strip()
            rt = table_to_text(val_dataset.samples[idx][1])
            pred, ref = parse_table(pt), parse_table(rt)
            empty += len(pred) == 0
            R.append(compare(pred, ref)); X.append(extra_metrics(pred, ref))
            saved.append({"image": os.path.basename(val_dataset.samples[idx][0]), "pred": pt, "ref": rt})
            idx += 1
    S = lambda k: sum(r[k] for r in R); T = lambda k: sum(x[k] for x in X)
    rec, prec = S("match") / max(S("n_ref"), 1), S("match") / max(S("n_pred"), 1)
    print(f"\n=== Validation [{tag}] | no_repeat_ngram={no_repeat_ngram} ===")
    print(f"Evaluated: {len(R)} / {len(val_dataset)} | empty predictions: {empty}")
    print(f"Exact table match      : {S('exact')/len(R):.4f}")
    print(f"Shape match / row-count: {S('shape')/len(R):.4f} / {S('rows')/len(R):.4f}")
    print(f"Cell accuracy (micro)  : {rec:.4f} | macro: {np.mean([r['match']/max(r['n_ref'],1) for r in R]):.4f}")
    print(f"Cell precision / F1    : {prec:.4f} / {2*prec*rec/max(prec+rec,1e-12):.4f}")
    print(f"Header accuracy        : {T('hm')/max(T('hn'),1):.4f}")
    print(f"First-column accuracy  : {T('em')/max(T('en'),1):.4f}")
    print(f"Numeric acc (±1% tol)  : {T('nm')/max(T('nn'),1):.4f}")
    json.dump(saved, open(f"{WORK}/val_predictions_{tag}.json", "w"))   # predictions kept for qualitative figures
    for s in saved[:2]:
        print(f"\n[{s['image']}]\nPRED: {s['pred'][:300]}\nREF : {s['ref'][:300]}")
    return R, saved

**Run the evaluation.** With the defaults it evaluates `EVAL_N = 32` charts (about 1 min), enough to check that everything works. Set `EVAL_N = None` for the full validation set (about 21 min) to reproduce the table in [Section 11](#s11). Predictions are saved to `/kaggle/working/val_predictions_<tag>.json`.

In [ ]:
R, saved = evaluate(f"v1_epoch{EVAL_EPOCH}_plain", no_repeat_ngram=0, n_eval=EVAL_N)    # main setting
# Optional ablation (repetition guard), about 21 min on the full set:
# R_g, saved_g = evaluate(f"v1_epoch{EVAL_EPOCH}_guard", no_repeat_ngram=8, n_eval=EVAL_N)

<a id="s11"></a>
## 11. Results  <small>[↑ top](#top)</small>

All numbers below were recorded from earlier full runs (1,054 validation charts, greedy decoding, positional matching). They are typed in as constants so the plots work without re-running the evaluation.

### 11.1 Loss history (per epoch)
| Epoch | Samples seen (cumulative) | Train loss | Val loss |
|---|---|---|---|
| 1 | 9,158 | 2.1170 | 2.0344 * |
| 2 | 27,474 | 1.5546 | 1.5469 |
| 3 | 45,790 | 1.1731 | 1.2578 |

\* validated in `train()` mode (dropout on). Per-step loss history was not saved for v1; only epoch-level values exist.

### 11.2 Generation metrics
| Metric | Epoch 1 (half data) | Epoch 2 | **Epoch 3** |
|---|---|---|---|
| Exact table match | 0.0000 | 0.0000 | **0.0009** |
| Shape match | 0.0389 | 0.1157 | **0.1641** |
| Row-count match | 0.0569 | 0.1442 | **0.2268** |
| Cell accuracy (micro) | 0.1246 | 0.2034 | **0.2516** |
| Cell accuracy (macro) | 0.1478 | 0.2415 | **0.3008** |
| Cell precision | 0.0379 | 0.0643 | **0.0892** |
| Cell F1 | 0.0581 | 0.0978 | **0.1317** |
| Header accuracy | n/a | n/a | **0.6094** |
| First-column accuracy | n/a | n/a | **0.3425** |
| Numeric accuracy (±1%) | n/a | n/a | **0.1396** |

(n/a: header, first-column and numeric accuracy were added after the epoch-1 and epoch-2 evaluations.)

In [ ]:
import matplotlib.pyplot as plt

# ---- Figure 1: epoch-level loss ----
x   = [9158, 27474, 45790]
tr  = [2.1170, 1.5546, 1.1731]
va  = [2.0344, 1.5469, 1.2578]
plt.figure(figsize=(6.4, 4))
plt.plot(x, tr, "o--", label="train"); plt.plot(x, va, "o-", label="validation")
for a, b in zip(x, va): plt.annotate(f"{b:.3f}", (a, b), textcoords="offset points", xytext=(0, 7), ha="center", fontsize=8)
plt.xlabel("cumulative training samples seen"); plt.ylabel("cross-entropy loss")
plt.title("v1: loss per epoch (epoch 1 used half the data)"); plt.legend(); plt.grid(alpha=.3)
plt.tight_layout(); plt.savefig(f"{WORK}/v1_fig_loss.png", dpi=200); plt.show()

# ---- Figure 2: metric progression across epochs ----
names = ["Shape", "Row-count", "Cell acc (micro)", "Cell acc (macro)", "Precision", "F1"]
ep = {1: [0.0389, 0.0569, 0.1246, 0.1478, 0.0379, 0.0581],
      2: [0.1157, 0.1442, 0.2034, 0.2415, 0.0643, 0.0978],
      3: [0.1641, 0.2268, 0.2516, 0.3008, 0.0892, 0.1317]}
xs, w = np.arange(len(names)), 0.27
plt.figure(figsize=(9.5, 4))
for k, e in enumerate((1, 2, 3)):
    bars = plt.bar(xs + (k - 1) * w, ep[e], w, label=f"epoch {e}")
    plt.bar_label(bars, fmt="%.2f", fontsize=7, padding=1)
plt.xticks(xs, names, rotation=15); plt.ylim(0, 0.4); plt.ylabel("score")
plt.title("v1: validation metrics by epoch (exact match: 0, 0, 0.0009)"); plt.legend(); plt.grid(axis="y", alpha=.3)
plt.tight_layout(); plt.savefig(f"{WORK}/v1_fig_metrics.png", dpi=200); plt.show()

### 11.3 Observations
* **Steady but slow improvement.** Every metric rises each epoch, but after 2.5 epochs the model still gets only about a quarter of the cells right and almost no complete tables (exact match 0.0009, i.e. 1 of 1,054 charts).
* **Layout is learned before values.** Headers are right 61% of the time and entity names 34%; numeric cells only 14%. The decoder learns the table *syntax* first and the harder visual-to-number mapping much later.
* **Precision ≪ recall** (0.089 vs 0.252): the model often writes far more cells than the chart contains because of repetition loops.
* **Typical failures seen in the Gradio demo:** a single value repeated down the whole table (`| 39 | 39 | 39 …`), the same entity repeated (`Finland` in every row), and the wrong number of columns (two-column output for a three-column chart). Many generations hit the 512-token limit while looping.
* **Why this happens (hypotheses, not tested here).** A decoder trained from scratch on about 18k charts at batch size 1 has to learn language modelling, number formatting and table syntax from this data alone. This motivates v2, where the decoder is initialised from Donut's pretrained decoder.
* **Validation loss is teacher-forced**, so it flatters free generation: a model can have low loss yet derail when it must feed on its own predictions.

<a id="s12"></a>
## 12. Inference app (Gradio)  <small>[↑ top](#top)</small>

A small web interface lets the user **upload a chart image** and see:
* the **raw generated sequence** (with ` | ` and ` [ROW] ` separators),
* the **parsed table** (first row used as header; ragged rows padded for display only),
* generation info (token count, row widths, whether it stopped at EOS or hit the 512-token limit),
* a **CSV download** of the parsed table, and a checkbox for the repetition guard (off by default).

The app is a standalone script (`inference.py`) that rebuilds the same architecture, loads a checkpoint, and uses the same KV-cached greedy decoder as this notebook. It supports both architectures; v1 is selected with `--arch v1`.

* Cell 1 writes the script to `/kaggle/working/inference.py`.
* Cell 2 launches it when `RUN_APP = True`. It prints a public `gradio.live` link; the cell keeps running while the app is up, so stop the cell to shut the app down. (Install Gradio first if needed: `%pip install -q gradio`.)

In [ ]:
%%writefile /kaggle/working/inference.py
"""
Chart-to-Table inference app (frozen Donut encoder + trainable decoder).

Architectures
  v2 (default): Donut's own PRETRAINED decoder (MBartForCausalLM), checkpoint decoder_v2_epoch1.pt
  v1          : 6-layer BART decoder trained from scratch, checkpoint epoch_3.pt

Usage (Kaggle notebook cell, GPU T4 + Internet On; keep the Kaggle default transformers version,
which is the one the v2 evaluation ran under):
    !pip install -q gradio
    !python /kaggle/working/inference.py                                  # v2, checkpoint auto-detected
    !python /kaggle/working/inference.py --ckpt /path/to/decoder_v2_epoch1.pt
    !python /kaggle/working/inference.py --arch v1 --ckpt /path/to/epoch_3.pt

Usage (local, GPU strongly recommended):
    python inference.py --ckpt path/to/decoder_v2_epoch1.pt

A checkpoint may hold "decoder_state_dict" (decoder only) or "model_state_dict" (full model).
The frozen Donut encoder is always downloaded from Hugging Face (naver-clova-ix/donut-base).
"""
import argparse
import glob
import os
import re
import tempfile

os.environ["TOKENIZERS_PARALLELISM"] = "false"

import gradio as gr
import pandas as pd
import torch
import torch.nn as nn
from transformers import BartConfig, BartForCausalLM, DonutProcessor, VisionEncoderDecoderModel

DONUT = "naver-clova-ix/donut-base"
ROW_SPLIT = re.compile(r"\s*\[ROW\]\s*")
CELL_SPLIT = re.compile(r"\s*\|\s*")
DEFAULT_CKPT = {"v2": "decoder_v2_epoch1.pt", "v1": "epoch_3.pt"}


# --------------------------------------------------------------------------- models
class DonutPretrainedDecoderModel(nn.Module):
    """v2: frozen Donut encoder + Donut's pretrained decoder (same setup as training)."""

    def __init__(self, donut, tokenizer):
        super().__init__()
        self.encoder = donut.encoder
        for p in self.encoder.parameters():
            p.requires_grad = False
        self.decoder = donut.decoder
        c = self.decoder.config
        assert c.vocab_size == len(tokenizer), (c.vocab_size, len(tokenizer))
        c.pad_token_id, c.bos_token_id, c.eos_token_id = (
            tokenizer.pad_token_id, tokenizer.bos_token_id, tokenizer.eos_token_id)
        c.decoder_start_token_id = tokenizer.bos_token_id
        c.use_cache = False


class DonutBartModel(nn.Module):
    """v1: frozen Donut encoder + 6-layer BART decoder trained from scratch."""

    def __init__(self, donut, tokenizer):
        super().__init__()
        self.encoder = donut.encoder
        for p in self.encoder.parameters():
            p.requires_grad = False
        cfg = BartConfig(
            vocab_size=len(tokenizer), d_model=self.encoder.config.hidden_size,
            decoder_layers=6, decoder_attention_heads=8, decoder_ffn_dim=4096,
            max_position_embeddings=1024, pad_token_id=tokenizer.pad_token_id,
            bos_token_id=tokenizer.bos_token_id, eos_token_id=tokenizer.eos_token_id,
            decoder_start_token_id=tokenizer.bos_token_id,
            is_decoder=True, add_cross_attention=True, use_cache=False,
        )
        self.decoder = BartForCausalLM(cfg)


def find_checkpoint(arch):
    name = DEFAULT_CKPT[arch]
    hits = glob.glob(f"/kaggle/working/{name}") + sorted(glob.glob(f"/kaggle/input/**/{name}", recursive=True))
    assert hits, f"--ckpt not given and {name} not found under /kaggle/working or /kaggle/input"
    return hits[0]


def load_model(ckpt_path, arch, device):
    processor = DonutProcessor.from_pretrained(DONUT)
    donut = VisionEncoderDecoderModel.from_pretrained(DONUT)
    cls = DonutPretrainedDecoderModel if arch == "v2" else DonutBartModel
    model = cls(donut, processor.tokenizer)

    ck = torch.load(ckpt_path, map_location="cpu", weights_only=False)
    if "model_state_dict" in ck:
        model.load_state_dict(ck["model_state_dict"])
    elif "decoder_state_dict" in ck:
        model.decoder.load_state_dict(ck["decoder_state_dict"])
    else:
        raise KeyError(f"Unrecognised checkpoint keys: {list(ck.keys())}")
    print(f"Loaded {ckpt_path} | arch: {arch} | val loss: {ck.get('val_loss', 'n/a')} "
          f"| samples seen: {ck.get('samples_seen', ck.get('epoch', 'n/a'))}")
    del ck
    return model.to(device).eval(), processor


# ----------------------------------------------------------------------- decoding
@torch.no_grad()
def generate(model, pixel_values, max_new_tokens=512, no_repeat_ngram=0):
    """KV-cached greedy decoding with cross-attention to the frozen encoder (identical to the
    decoder used for the validation results). no_repeat_ngram > 1 bans repeated n-grams.
    Returns (token_ids, hit_eos)."""
    cfg = model.decoder.config
    hs = model.encoder(pixel_values).last_hidden_state
    cur = torch.full((1, 1), cfg.decoder_start_token_id, device=pixel_values.device)
    past, hist, seen, ids, hit_eos = None, [], {}, [], False
    n = no_repeat_ngram
    for _ in range(max_new_tokens):
        out = model.decoder(input_ids=cur, encoder_hidden_states=hs,
                            past_key_values=past, use_cache=True, return_dict=True)
        past = out.past_key_values
        logits = out.logits[:, -1].float()
        if n > 1 and len(hist) >= n - 1:
            banned = seen.get(tuple(hist[-(n - 1):]))
            if banned:
                logits[0, list(banned)] = float("-inf")
        nxt = int(logits.argmax(-1))
        if n > 1 and len(hist) >= n - 1:
            seen.setdefault(tuple(hist[-(n - 1):]), set()).add(nxt)
        hist.append(nxt)
        if nxt == cfg.eos_token_id:
            hit_eos = True
            break
        ids.append(nxt)
        cur = torch.tensor([[nxt]], device=pixel_values.device)
    return ids, hit_eos


# ----------------------------------------------------------------------- parsing
def parse_table(text):
    text = text.strip()
    return [CELL_SPLIT.split(r.strip()) for r in ROW_SPLIT.split(text)] if text else []


def to_dataframe(rows):
    """Pad ragged rows for display only; the first predicted row is used as the header."""
    if not rows:
        return pd.DataFrame()
    width = max(map(len, rows))
    padded = [r + [""] * (width - len(r)) for r in rows]
    header, seen = [], {}
    for j, h in enumerate(padded[0]):
        h = h or f"col{j + 1}"
        seen[h] = seen.get(h, 0) + 1
        header.append(h if seen[h] == 1 else f"{h} ({seen[h]})")
    return pd.DataFrame(padded[1:], columns=header)


# --------------------------------------------------------------------------- app
def build_app(model, processor, device, arch):
    tok = processor.tokenizer

    def run(image, use_guard, max_tokens):
        if image is None:
            return "Upload a chart image first.", pd.DataFrame(), "", None
        pv = processor(image.convert("RGB"), return_tensors="pt").pixel_values.to(device)
        ctx = (torch.autocast("cuda", dtype=torch.float16) if device.type == "cuda"
               else torch.autocast("cpu", enabled=False))
        with ctx:
            ids, hit_eos = generate(model, pv, int(max_tokens), no_repeat_ngram=8 if use_guard else 0)
        text = tok.decode(ids, skip_special_tokens=True).strip()
        rows = parse_table(text)
        df = to_dataframe(rows)
        widths = sorted({len(r) for r in rows})
        info = (f"{len(ids)} tokens | {len(rows)} rows | row widths: {widths or '-'} | "
                + ("stopped at EOS" if hit_eos else "hit the token limit (output may be looping or cut off)"))
        csv_path = None
        if not df.empty:
            tmp = tempfile.NamedTemporaryFile(suffix=".csv", delete=False)
            df.to_csv(tmp.name, index=False)
            csv_path = tmp.name
        return text, df, info, csv_path

    with gr.Blocks(title="Chart-to-Table") as demo:
        gr.Markdown(
            f"## Chart → Table (frozen Donut encoder, architecture {arch})\n"
            "Upload a chart image. The raw output uses ` | ` between cells and ` [ROW] ` between rows; "
            "the table below is parsed from it. Large tables and exact numeric values are still the "
            "weakest part of the model, so check values against the chart.")
        with gr.Row():
            with gr.Column():
                img = gr.Image(type="pil", label="Chart image")
                guard = gr.Checkbox(
                    value=False,
                    label="Repetition guard (block repeated 8-grams). Lowered accuracy in validation; leave off")
                max_tok = gr.Slider(64, 512, value=512, step=32, label="Max generated tokens")
                btn = gr.Button("Generate table", variant="primary")
            with gr.Column():
                raw = gr.Textbox(label="Raw generated sequence", lines=6)
                table = gr.Dataframe(label="Parsed table (first row = header)", interactive=False)
                stats = gr.Textbox(label="Generation info")
                csv_file = gr.File(label="Download parsed table (CSV)")
        btn.click(run, [img, guard, max_tok], [raw, table, stats, csv_file])
    return demo


if __name__ == "__main__":
    ap = argparse.ArgumentParser()
    ap.add_argument("--arch", choices=["v2", "v1"], default="v2")
    ap.add_argument("--ckpt", default=None, help="checkpoint path (auto-detected if omitted)")
    ap.add_argument("--no-share", action="store_true", help="disable the public Gradio link")
    args, _ = ap.parse_known_args()

    ckpt = args.ckpt or find_checkpoint(args.arch)
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    print("Device:", device)
    model, processor = load_model(ckpt, args.arch, device)
    build_app(model, processor, device, args.arch).launch(share=not args.no_share, debug=False)

In [ ]:
if RUN_APP:
    assert CKPT[EVAL_EPOCH], "checkpoint not found"
    get_ipython().system(f"python /kaggle/working/inference.py --arch v1 --ckpt {CKPT[EVAL_EPOCH]}")
else:
    print("RUN_APP is False -> app not launched. Set RUN_APP = True in Section 2 to start it.")

<a id="s13"></a>
## 13. Reproducibility notes & caveats  <small>[↑ top](#top)</small>

**Files produced / needed**
| File | Content | Size |
|---|---|---|
| `epoch_1.pt`, `epoch_2.pt`, `epoch_3.pt` | encoder + decoder weights, optimizer and scaler state, recorded losses | ≈ 2.2 GB each |
| `ckpt_latest.pt`, `ckpt3_latest.pt` | mid-epoch resume files (deleted when the epoch finishes) | ≈ 1.5 GB |
| `val_predictions_<tag>.json` | generated vs reference table text for each evaluated chart | small |
| `v1_fig_loss.png`, `v1_fig_metrics.png` | figures from Section 11 | small |

**Reproducibility**
* Seeds: epoch-1 subset selection seed 42 (the loader shuffle itself was unseeded); epoch 2 seed 2; epoch 3 seed 3. Training is **not bit-exact reproducible** (fp16, non-deterministic CUDA kernels).
* Environment is listed in [Section 1](#s1).
* Data: 18,316 train / 1,054 validation charts. The test split is **not evaluated** in this notebook.

**Caveats to state when reporting**
1. Single run, single seed, greedy decoding only; no confidence intervals.
2. Epoch 1 used half the training data and a validation loss computed with dropout on; epochs are therefore not perfectly comparable.
3. Train loss is a running mean over a moving model, so it is not directly comparable with validation loss.
4. Positional matching is strict; a bag-of-cells score would be higher.
5. Six validation references exceed the 512-token target length.
6. Header, first-column and numeric metrics exist only for epoch 3.

**Next.** The v2 notebook keeps the frozen encoder and initialises the decoder from Donut's pretrained decoder (with effective batch size 8 and an LR schedule); it improves every metric above and is the baseline for the weakly-supervised comparison.